# Regression

Regression estimates relationships between variables. In finance, it is often used to estimate beta, factor exposure, and return drivers.

Abbreviations used in this notebook:

- **OLS**: Ordinary Least Squares.
- **CAPM**: Capital Asset Pricing Model.
- **R²**: R-squared, the share of variation explained by the model.
- **MSE**: Mean Squared Error.
- **RMSE**: Root Mean Squared Error.
- **SSE**: Sum of Squared Errors.
- **SST**: Total Sum of Squares.

## 1. Intuition

Regression asks how one variable tends to move when another variable changes. For a stock, market regression estimates beta: how sensitive the stock is to broad market moves.

## 2. Mathematics

Simple regression:

$$
y_t = \alpha + \beta x_t + \epsilon_t
$$

OLS minimizes squared errors:

$$
\min_{\alpha,\beta} \sum_t (y_t - \hat{y}_t)^2
$$

CAPM regression:

$$
R_i - R_f = \alpha + \beta(R_m - R_f) + \epsilon
$$

R-squared:

$$
R^2 = 1 - \frac{SSE}{SST}
$$

Where:
- `y_t` = dependent variable at time `t`.
- `x_t` = explanatory variable at time `t`.
- `alpha` = intercept or abnormal return.
- `beta` = sensitivity to the explanatory variable or market factor.
- `epsilon_t` = residual error term.
- `R_i` = asset return.
- `R_m` = market return.
- `R_f` = risk-free return.
- `SSE` = sum of squared errors.
- `SST` = total sum of squares.


## 3. Implementation

We estimate market beta for a synthetic asset using OLS implemented with numpy.

In [ ]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "GUIDELINES.md").exists())
helper_path = project_root / "04_quantitative_methods" / "quant_utils.py"
spec = importlib.util.spec_from_file_location("quant_utils", helper_path)
quant_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(quant_utils)

plt.style.use("seaborn-v0_8-whitegrid")
returns = quant_utils.generate_return_sample()

excess_asset = returns["asset"] - returns["risk_free"]
excess_market = returns["market"] - returns["risk_free"]
model = quant_utils.ordinary_least_squares(excess_asset.to_numpy(), excess_market.to_numpy())
alpha, beta = model["coefficients"]

regression_summary = pd.Series({
    "daily_alpha": alpha,
    "annualized_alpha_approx": alpha * 252,
    "market_beta": beta,
    "r_squared": model["r_squared"],
    "rmse": np.sqrt(np.mean(model["residuals"] ** 2)),
})
regression_summary.to_frame("value")

In [ ]:
regression_frame = pd.DataFrame({
    "excess_market": excess_market,
    "excess_asset": excess_asset,
    "fitted": model["fitted"],
    "residual": model["residuals"],
})
regression_frame.head()

## 4. Visualization

A scatter plot with a fitted line shows the estimated relationship; residual plots show what the model misses.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(regression_frame["excess_market"], regression_frame["excess_asset"], alpha=0.35, color="#2f6f8f", s=14)
x_line = np.linspace(regression_frame["excess_market"].min(), regression_frame["excess_market"].max(), 100)
y_line = alpha + beta * x_line
axes[0].plot(x_line, y_line, color="#9a6b2f", linewidth=2)
axes[0].set_title("CAPM Regression")
axes[0].set_xlabel("Market excess return")
axes[0].set_ylabel("Asset excess return")

axes[1].hist(regression_frame["residual"], bins=35, color="#2f6f8f", edgecolor="white")
axes[1].set_title("Residual Distribution")
axes[1].set_xlabel("Residual")
plt.tight_layout(); plt.show()

## 5. Application

Regression helps estimate beta for cost of equity, factor exposure for portfolio risk, and sensitivity to macro variables.

In [ ]:
stress_market_return = -0.03
predicted_asset_return = alpha + beta * stress_market_return
print(f"Estimated beta: {beta:.2f}")
print(f"Predicted asset excess return if market excess return is -3%: {predicted_asset_return:.2%}")

## 6. Reflection

- Beta is an estimate, not a permanent truth.
- High R-squared means the model explains more variation, not that it predicts perfectly.
- Residuals are the part the model does not explain.
- Regression relationships can break when regimes change.

Questions to answer after running the notebook:

1. Is the asset more or less sensitive than the market?
2. What does alpha mean in this regression?
3. Why inspect residuals?
4. What would make beta unstable?